In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    StratifiedGroupKFold,
    train_test_split
)

# --------------------------------------------------
# 1. Load the feature-engineered dataset
# --------------------------------------------------

df = pd.read_csv("AI_Fresher_Recruitment_ML_Dataset.csv")

target = "Shortlisted"
leakage_columns = [
    "Recruitment_Score",
    "Rank",
    "Shortlist Limit",
    "Shortlisted"
]

# Target
X = df.drop(columns=leakage_columns)
y = df["Shortlisted"]

# Candidate ID is used only for grouping
groups = df["Candidate ID"]


# --------------------------------------------------
# 2. Create a candidate-level test set
# --------------------------------------------------

# Get one row per unique candidate
candidate_info = (
    df[["Candidate ID", "Shortlisted"]]
    .groupby("Candidate ID")["Shortlisted"]
    .max()
    .reset_index()
)

# Split CANDIDATES, not individual application rows
train_candidates, test_candidates = train_test_split(
    candidate_info["Candidate ID"],
    test_size=0.15,
    random_state=42,
    stratify=candidate_info["Shortlisted"]
)

# Convert to sets for easy filtering
train_candidates = set(train_candidates)
test_candidates = set(test_candidates)


# --------------------------------------------------
# 3. Create train and test datasets
# --------------------------------------------------

train_val_df = df[
    df["Candidate ID"].isin(train_candidates)
].copy()

test_df = df[
    df["Candidate ID"].isin(test_candidates)
].copy()


# --------------------------------------------------
# 4. Separate X and y
# --------------------------------------------------

X_train_val = train_val_df.drop(
    columns=["Shortlisted"]
)

y_train_val = train_val_df["Shortlisted"]

groups_train_val = train_val_df["Candidate ID"]

X_test = test_df.drop(
    columns=["Shortlisted"]
)

y_test = test_df["Shortlisted"]


# --------------------------------------------------
# 5. 5-Fold Stratified Group Cross-Validation
# --------------------------------------------------

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# --------------------------------------------------
# 6. Check the folds
# --------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(
        X_train_val,
        y_train_val,
        groups=groups_train_val
    ),
    start=1
):

    X_train = X_train_val.iloc[train_idx]
    X_val = X_train_val.iloc[val_idx]

    y_train = y_train_val.iloc[train_idx]
    y_val = y_train_val.iloc[val_idx]

    print(f"\nFold {fold}")

    print("Training applications:", len(X_train))
    print("Validation applications:", len(X_val))

    print(
        "Training candidates:",
        X_train_val.iloc[train_idx]["Candidate ID"].nunique()
    )

    print(
        "Validation candidates:",
        X_train_val.iloc[val_idx]["Candidate ID"].nunique()
    )

    print(
        "Training shortlisted:",
        y_train.sum()
    )

    print(
        "Validation shortlisted:",
        y_val.sum()
    )


Fold 1
Training applications: 11247
Validation applications: 2792
Training candidates: 6802
Validation candidates: 1698
Training shortlisted: 150
Validation shortlisted: 30

Fold 2
Training applications: 11221
Validation applications: 2818
Training candidates: 6801
Validation candidates: 1699
Training shortlisted: 147
Validation shortlisted: 33

Fold 3
Training applications: 11202
Validation applications: 2837
Training candidates: 6802
Validation candidates: 1698
Training shortlisted: 137
Validation shortlisted: 43

Fold 4
Training applications: 11270
Validation applications: 2769
Training candidates: 6798
Validation candidates: 1702
Training shortlisted: 143
Validation shortlisted: 37

Fold 5
Training applications: 11216
Validation applications: 2823
Training candidates: 6797
Validation candidates: 1703
Training shortlisted: 143
Validation shortlisted: 37
